# EDA — Capa Gold (Sprint 1)

Este notebook consulta las 4 tablas Gold y responde las preguntas analíticas B-1..B-4 del proyecto.

Correr dentro del contenedor `spark-iceberg`:

    docker compose exec spark-iceberg jupyter lab --ip=0.0.0.0 --no-browser

y abrir http://localhost:8888.


In [ ]:
import sys
sys.path.insert(0, '/workspace/src')

from shared.config import (
    crear_spark_session,
    TBL_GOLD_AFLUENCIA_PM25,
    TBL_GOLD_ACCIDENTALIDAD,
    TBL_GOLD_ENCICLA_CLIMA,
    TBL_GOLD_CORREDORES_RIESGO,
)

spark = crear_spark_session('EDA-Gold')
spark.sparkContext.setLogLevel('ERROR')

## B-1 · Correlación PM2.5 / lluvia — afluencia Metro


In [ ]:
import matplotlib.pyplot as plt

df = spark.table(TBL_GOLD_AFLUENCIA_PM25).toPandas()
print(f'Filas: {len(df):,}')
df.head(10)

In [ ]:
# Correlación promedio por línea
g = df.groupby('linea')[['corr_pm25_validaciones','corr_precip_validaciones']].mean()
print(g)
g.plot(kind='bar', title='Correlación promedio (Pearson) por línea Metro')
plt.ylabel('Pearson')
plt.tight_layout()
plt.show()

## B-2 · Accidentalidad por comuna


In [ ]:
df2 = spark.table(TBL_GOLD_ACCIDENTALIDAD).toPandas()
ult = df2['anio'].max()
top10 = df2[df2['anio']==ult].nsmallest(10, 'ranking_severidad')[['comuna','con_heridos','con_muertos','solo_danos','indice_severidad']]
print(f'Top 10 comunas más severas en {ult}:')
print(top10.to_string(index=False))
ax = top10.set_index('comuna')['indice_severidad'].plot(kind='barh', title=f'Índice de severidad por comuna · {ult}')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## B-3 · Demanda EnCicla vs clima


In [ ]:
df3 = spark.table(TBL_GOLD_ENCICLA_CLIMA).toPandas()
# Comparar promedio de viajes por bin temperatura
g = df3[df3['bin_temperatura']!='sin_dato'].groupby('bin_temperatura')['viajes'].mean().sort_index()
print(g)
g.plot(kind='bar', title='Viajes promedio EnCicla por rango de temperatura (°C)')
plt.ylabel('Viajes / día')
plt.tight_layout()
plt.show()

In [ ]:
# Efecto de la lluvia
g2 = df3.groupby('llovio')['viajes'].mean()
print(f'Promedio sin lluvia: {g2[0]:.1f}, con lluvia: {g2[1]:.1f}')
print(f'Caída relativa con lluvia: {(g2[0]-g2[1])/g2[0]*100:.1f}%')

## B-4 · Corredores de riesgo compuesto


In [ ]:
df4 = spark.table(TBL_GOLD_CORREDORES_RIESGO).toPandas().sort_values('score_riesgo').head(15)
print(df4[['comuna','vehiculos_total','con_muertos','con_heridos','score_riesgo']].to_string(index=False))
df4.plot(x='comuna', y='score_riesgo', kind='barh', title='Top 15 comunas por riesgo compuesto (menor=peor)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()